# Yandex AI Studio: музейный экспонат от текста до голоса и изображения

В этом демонстрационном ноутбуке мы пройдём полный мультимодальный сценарий на данных
из открытой онлайн-коллекции Государственного музея изобразительных искусств имени
А. С. Пушкина. Один и тот же экспонат будет сопровождать нас на всех этапах, поэтому
результат каждой модели можно будет сравнить с исходными музейными данными.

Мы последовательно:

1. подключимся к Yandex AI Studio через OpenAI-совместимый Responses API;
2. сделаем простой запрос к LLM и продолжим диалог с сохранением контекста;
3. загрузим локальный музейный датасет и адаптируем описание для ребёнка 10 лет;
4. извлечём из полного текста типизированные смысловые теги;
5. передадим изображение экспоната мультимодальной модели (VLM), сначала получив обычный текст, а затем структурированный результат;
6. озвучим доступное визуальное описание с помощью SpeechKit;
7. создадим новую художественную интерпретацию экспоната с помощью YandexART.

> Модельные ответы вероятностны: формулировки будут немного отличаться при каждом запуске.
> Выводы LLM и VLM — не замена научной атрибуции музейного специалиста.

## 0. Подготовка окружения

Для выполнения примеров понадобятся OpenAI-совместимый клиент, нативный SDK Yandex AI
Studio (для SpeechKit), Pydantic для структурного вывода, Pillow для изображений,
`requests` для загрузки фотографии и `python-dotenv` для чтения секретов.

Следующая ячейка устанавливает или обновляет эти библиотеки в окружении текущего ядра.
После первого выполнения может потребоваться перезапустить kernel и снова начать с этой
секции. Ключи доступа в команду установки не передаются.

In [ ]:
import subprocess
import sys

packages = [
    "openai",
    "python-dotenv",
    "pydantic",
    "Pillow",
    "requests",
    "yandex-ai-studio-sdk",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *packages]
)
print("Зависимости установлены. Если импорты ниже не сработают, перезапустите kernel.")

Теперь импортируем только те компоненты, которые используются дальше. `IPython.display`
нужен для показа музейной фотографии, воспроизведения WAV-файла и отображения
сгенерированной иллюстрации непосредственно в ноутбуке.

In [ ]:
import base64
import io
import json
import os
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import Audio, JSON, Markdown, display
from openai import OpenAI
from PIL import Image
from pydantic import BaseModel, Field
from yandex_ai_studio_sdk import AIStudio

## 1. Подключение к Yandex Cloud

Как и в учебном примере `CloudConnect.ipynb`, идентификатор каталога `folder_id` и
API-ключ `api_key` берутся из файла `.env`, а не записываются в ноутбук. Поместите `.env`
в корень проекта в таком виде:

```text
folder_id=ваш_идентификатор_каталога
api_key=ваш_api_ключ
```

Код ищет корень проекта по файлу `data/pushkin.json`. Поэтому ноутбук можно открыть как
из корня репозитория, так и из каталога `notebooks`. В выводе показывается только начало
идентификатора каталога; сам API-ключ никогда не печатается.

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "pushkin.json").exists():
            return candidate
    raise FileNotFoundError(
        "Не найден data/pushkin.json. Запустите ноутбук внутри репозитория."
    )


project_root = find_project_root()
env_path = project_root / ".env"
load_dotenv(env_path)

folder_id = os.getenv("folder_id")
api_key = os.getenv("api_key")
if not folder_id or not api_key:
    raise RuntimeError(
        f"Добавьте folder_id и api_key в файл {env_path} и повторите ячейку."
    )

print(f"Корень проекта: {project_root}")
print(f"Folder ID: {folder_id[:8]}…")

Создадим единый объект `client` для всех вызовов Responses API и Images API. Параметр
`base_url` направляет стандартный OpenAI SDK в Yandex AI Studio, а `project=folder_id`
связывает запросы с каталогом Yandex Cloud.

Здесь также явно задаются URI трёх моделей:

- `qwen3_model` — текстовые запросы и структурный вывод;
- `qwen36_model` — запросы с изображением;
- `yandex_art_model` — генерация изображения в демонстрации YandexART.

На дату подготовки ноутбука YandexART 2.0 объявлен к снятию с эксплуатации 18 августа
2026 года. После этой даты для новых проектов замените его на актуальную модель генерации
изображений из каталога Yandex AI Studio (например, Alice AI ART).

In [ ]:
client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id,
)

qwen3_model = f"gpt://{folder_id}/qwen3-235b-a22b-fp8"
qwen36_model = f"gpt://{folder_id}/qwen3.6-35b-a3b"
yandex_art_model = f"art://{folder_id}/yandex-art-2.0"

print("Клиент создан, URI моделей подготовлены.")

## 2. Простой вызов LLM и продолжение диалога

Начнём с короткого запроса, не связанного с датасетом: попросим модель рассказать добрый
анекдот про искусство. Аргумент `instructions` задаёт роль и ограничения ответа, а `input`
содержит реплику пользователя. `store=True` сохраняет ответ на стороне API, чтобы следующий
запрос мог сослаться на него по `response.id`.

In [ ]:
joke_response = client.responses.create(
    model=qwen3_model,
    store=True,
    instructions=(
        "Ты — доброжелательный музейный ведущий. Отвечай по-русски, кратко, "
        "без грубости и сомнительных исторических фактов."
    ),
    input="Расскажи анекдот про искусство.",
)

display(Markdown(joke_response.output_text))
print(f"ID ответа: {joke_response.id}")

Теперь продолжим именно этот диалог фразой «А про котиков?». Вместо повторной передачи
всей истории укажем `previous_response_id`. Модель получит контекст предыдущей шутки и
поймёт, что нужно рассказать ещё один анекдот — теперь про котиков и искусство.

In [ ]:
cats_response = client.responses.create(
    model=qwen3_model,
    store=True,
    previous_response_id=joke_response.id,
    input="А про котиков?",
)

display(Markdown(cats_response.output_text))

## 3. Чтение музейного датасета и адаптация описания

Упрощённый датасет находится в `data/pushkin.json` и представляет собой JSON-массив.
В каждой записи есть название, тип объекта, страна, период, материал, полное музейное
описание, URL изображения и служебные поля. Загрузим все записи, затем — как требует
сквозной сценарий — сохраним первый экспонат в переменной `exhibit = exhibits[0]`.

Обратите внимание: данные и изображения остаются связанными с первоисточником через поля
`source_url` и `image_url`. Для публикации или массового повторного использования материалов
отдельно проверьте условия правообладателя на странице музея.

In [ ]:
dataset_path = project_root / "data" / "pushkin.json"
with dataset_path.open("r", encoding="utf-8") as file:
    exhibits = json.load(file)

exhibit = exhibits[0]

print(f"Всего экспонатов: {len(exhibits)}")
display(
    JSON(
        {
            "id": exhibit["id"],
            "title": exhibit["title"],
            "type": exhibit["type"],
            "country": exhibit["country"],
            "period": exhibit["period"],
            "material": exhibit["material"],
            "inventory_number": exhibit["inventory_number"],
            "source_url": exhibit["source_url"],
        },
        expanded=True,
    )
)
print(f"Длина полного описания: {len(exhibit['description'])} символов")

Музейное описание рассчитано на взрослого читателя и содержит много исторических деталей.
Попросим LLM сократить его и объяснить экспонат ребёнку 10 лет. В инструкции явно запретим
придумывать факты и попросим сохранить важные сведения: кто изображён, когда и где создан
предмет, из чего он сделан и почему интересен.

Полученный текст сохраним как `child_description`: он пригодится как самостоятельная
детская этикетка, хотя перед публикацией её всё равно должен проверить редактор музея.

In [ ]:
child_prompt = {
    "title": exhibit["title"],
    "country": exhibit["country"],
    "period": exhibit["period"],
    "material": exhibit["material"],
    "description": exhibit["description"],
}

child_response = client.responses.create(
    model=qwen3_model,
    instructions=(
        "Ты — редактор детского музея. Перепиши описание для ребёнка 10 лет. "
        "Используй простой русский язык, короткие предложения и 2–3 небольших абзаца. "
        "Сохрани только важные факты из источника, ничего не выдумывай. "
        "Если встречается сложный термин, сразу объясни его простыми словами."
    ),
    input=json.dumps(child_prompt, ensure_ascii=False),
)

child_description = child_response.output_text.strip()
display(Markdown(f"### Детская этикетка\n\n{child_description}"))

## 4. Структурный вывод: смысловые теги из полного текста

Свободный текст удобен читателю, но приложениям часто нужны поля фиксированной формы.
Опишем результат классом Pydantic `ExhibitSemanticTags`. Он потребует от модели вернуть
эпоху, художественный стиль, техники/материалы и набор ключевых слов. Дополнительное поле
`evidence` позволит кратко объяснить, на каких фрагментах исходного описания основана
классификация.

Если стиль в источнике прямо не назван, модель должна написать «не указан», а не угадывать.
Это особенно важно для музейных данных, где правдоподобная догадка может оказаться ложной.

In [ ]:
class ExhibitSemanticTags(BaseModel):
    epoch: str = Field(description="Эпоха или исторический период")
    style: str = Field(description="Художественный стиль либо 'не указан'")
    techniques: list[str] = Field(
        min_length=1,
        description="Материалы и техники изготовления, названные в источнике",
    )
    keywords: list[str] = Field(
        min_length=3,
        max_length=12,
        description="Короткие предметные теги без повторов",
    )
    evidence: list[str] = Field(
        min_length=1,
        max_length=4,
        description="Краткие основания из исходного описания",
    )

Метод `client.responses.parse` передаёт схему модели и возвращает уже проверенный объект
`ExhibitSemanticTags` в `output_parsed`. В запрос отправляется именно **полное** музейное
описание текущего `exhibit`, а не сокращённая детская версия. Для удобства просмотра
преобразуем типизированный объект обратно в JSON только на этапе отображения.

In [ ]:
tags_response = client.responses.parse(
    model=qwen3_model,
    instructions=(
        "Извлеки сведения только из переданного музейного текста. "
        "Не достраивай эпоху или стиль по общим знаниям. "
        "Формулируй значения по-русски и делай теги пригодными для поиска."
    ),
    input=exhibit["description"],
    text_format=ExhibitSemanticTags,
)

semantic_tags = tags_response.output_parsed
display(JSON(semantic_tags.model_dump(), expanded=True))

## 5. VLM: анализ изображения экспоната

До вызова мультимодальной модели обязательно посмотрим на то же изображение сами. Ячейка
загружает `exhibit["image_url"]`, проверяет HTTP-ответ, преобразует данные в `PIL.Image`
и показывает фотографию вместе с музейным названием. Изображение пока никуда в модель
не отправляется — это отдельный прозрачный шаг подготовки входных данных.

In [ ]:
image_response = requests.get(exhibit["image_url"], timeout=30)
image_response.raise_for_status()
exhibit_image = Image.open(io.BytesIO(image_response.content)).convert("RGB")

display(Markdown(f"### {exhibit['title']}"))
display(exhibit_image)
print(f"Размер изображения: {exhibit_image.width} × {exhibit_image.height} пикселей")
print(f"Источник: {exhibit['source_url']}")

Responses API принимает изображение как URL или как data URL. Чтобы пример не зависел от
повторного доступа модели к внешнему сайту, преобразуем уже загруженный `PIL.Image` в JPEG,
кодируем байты в Base64 и сформируем `data:image/jpeg;base64,...`. Функция пригодится в обоих
следующих VLM-вызовах.

In [ ]:
def image_to_data_url(image: Image.Image) -> str:
    buffer = io.BytesIO()
    image.convert("RGB").save(buffer, format="JPEG", quality=92)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/jpeg;base64,{encoded}"


exhibit_image_data_url = image_to_data_url(exhibit_image)
print(f"Изображение подготовлено для VLM: {len(exhibit_image_data_url):,} символов")

Сначала попросим VLM дать обычное подробное описание видимого. В запрос намеренно не
передаются название, эпоха и музейный текст: так можно увидеть, что модель извлекает именно
из фотографии. Инструкция требует отделять наблюдаемое от предположений и не называть людей
или датировку без визуальных оснований.

Ответ сохраним в `visual_description`. Его объём ограничен примерно 700 символами, чтобы
следующая демонстрация синтеза речи оставалась короткой и удобной для прослушивания.

In [ ]:
vlm_response = client.responses.create(
    model=qwen36_model,
    instructions=(
        "Ты — специалист по тифлокомментированию музейных предметов. "
        "Опиши по-русски только то, что видно на фотографии: объект, позы, одежду, "
        "материалы, цвета, фактуру, композицию и заметные детали. "
        "Сначала дай общее впечатление, затем двигайся слева направо и сверху вниз. "
        "Не угадывай имена, точную эпоху или назначение. Объём — 500–700 символов."
    ),
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Составь подробное визуальное описание экспоната.",
                },
                {
                    "type": "input_image",
                    "image_url": exhibit_image_data_url,
                    "detail": "auto",
                },
            ],
        }
    ],
)

visual_description = vlm_response.output_text.strip()
display(Markdown(f"### Визуальное описание\n\n{visual_description}"))

Для структурного VLM-анализа создадим отдельную схему `VisualExhibitAnalysis`. Поля похожи
на текстовые смысловые теги, но теперь основанием служит только фотография. В схему также
входят композиция, доминирующие цвета, уровень уверенности и доступное описание для
озвучивания.

Поля эпохи, стиля и техники здесь являются **визуальными гипотезами**, а не атрибуцией.
Их полезно сравнить с `semantic_tags`, извлечёнными из проверенного музейного текста.

In [ ]:
class VisualExhibitAnalysis(BaseModel):
    detailed_description: str = Field(
        description="Связное описание только видимых особенностей"
    )
    composition: str = Field(description="Расположение основных форм и фигур")
    dominant_colors: list[str] = Field(min_length=1, max_length=8)
    likely_epoch: str = Field(description="Визуальная гипотеза об эпохе")
    likely_style: str = Field(description="Визуальная гипотеза о стиле")
    likely_techniques: list[str] = Field(
        min_length=1,
        description="Предполагаемые по изображению материалы или техники",
    )
    visual_keywords: list[str] = Field(min_length=3, max_length=12)
    confidence: str = Field(description="низкая, средняя или высокая")
    accessible_description: str = Field(
        max_length=900,
        description="Самодостаточное описание для озвучивания, не более 900 символов",
    )

Повторим передачу того же `exhibit_image_data_url`, но теперь через
`client.responses.parse`. В инструкции подчеркнём неопределённость визуальной атрибуции.
Результат `visual_analysis` — полноценный объект Pydantic: его можно валидировать, сохранять
в базу или использовать дальше без ручного разбора JSON.

In [ ]:
visual_tags_response = client.responses.parse(
    model=qwen36_model,
    instructions=(
        "Анализируй только изображение, без внешних знаний об объекте. "
        "Отделяй наблюдение от гипотезы. Если эпоху, стиль или технику нельзя "
        "надёжно определить по фотографии, явно укажи неопределённость и низкую уверенность. "
        "Все значения верни по-русски."
    ),
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Верни структурированный визуальный анализ этого экспоната.",
                },
                {
                    "type": "input_image",
                    "image_url": exhibit_image_data_url,
                    "detail": "auto",
                },
            ],
        }
    ],
    text_format=VisualExhibitAnalysis,
)

visual_analysis = visual_tags_response.output_parsed
display(JSON(visual_analysis.model_dump(), expanded=True))

## 6. Синтез речи: доступное описание экспоната

Для посетителя, которому трудно рассмотреть фотографию или прочитать текст, превратим поле
`accessible_description` в речь. SpeechKit вызывается через нативный `AIStudio` SDK, но
использует те же `folder_id` и `api_key`. Выберем русский голос `jane`, формат WAV и немного
замедленный темп, удобный для музейного аудиогида.

Аудиофайл будет сохранён в `outputs/exhibit-description.wav`, а встроенный проигрыватель
появится под ячейкой. В демонстрационном сценарии текст заранее ограничен 900 символами —
для длинной экскурсии его следует делить на смысловые фрагменты.

In [ ]:
sdk = AIStudio(folder_id=folder_id, auth=api_key)
output_dir = project_root / "outputs"
output_dir.mkdir(exist_ok=True)

tts_text = visual_analysis.accessible_description
tts = sdk.speechkit.text_to_speech(voice="jane", audio_format="WAV")
speech_result = tts.configure(speed=0.95).run(tts_text)

audio_path = output_dir / "exhibit-description.wav"
audio_path.write_bytes(speech_result.data)

print(f"Озвученный текст ({len(tts_text)} символов):\n{tts_text}\n")
print(f"Файл сохранён: {audio_path}")
display(Audio(filename=str(audio_path)))

## 7. Художественная интерпретация с YandexART

Визуальное описание написано для человека, а генератору изображений полезнее компактный
режиссёрский промпт. Попросим текстовую модель превратить `visual_description` в описание
сцены длиной не более 500 символов. В промпте потребуем музейную подачу и запретим текст,
логотипы и попытку выдать результат за документальную фотографию оригинала.

Это промежуточный, наблюдаемый шаг: перед генерацией промпт будет напечатан, и куратор сможет
отредактировать его вручную.

In [ ]:
art_prompt_response = client.responses.create(
    model=qwen3_model,
    instructions=(
        "Ты — арт-директор. Преобразуй входное визуальное описание в один русский промпт "
        "для генерации художественной музейной иллюстрации. Укажи объект, композицию, "
        "материалы, свет, фон и настроение. Не добавляй надписи, логотипы и новые объекты. "
        "Подчеркни, что это современная художественная интерпретация, а не копия. "
        "Верни только промпт длиной не более 500 символов."
    ),
    input=visual_description,
)

art_prompt = art_prompt_response.output_text.strip()
if len(art_prompt) > 500:
    raise ValueError(f"Промпт слишком длинный: {len(art_prompt)} символов")

print(f"Промпт для YandexART ({len(art_prompt)} символов):\n{art_prompt}")

Теперь вызовем OpenAI-совместимый Images API с моделью YandexART 2.0. Ответ содержит
изображение в Base64: декодируем его, откроем через Pillow, сохраним как JPEG в
`outputs/yandexart-exhibit.jpg` и покажем рядом с результатами предыдущих этапов.

Сгенерированная картинка — новая иллюстрация на основе машинного описания. Её нельзя
использовать как свидетельство внешнего вида, сохранности или атрибуции музейного оригинала.

In [ ]:
generated_response = client.images.generate(
    model=yandex_art_model,
    prompt=art_prompt,
    n=1,
    size="1024x1024",
)

generated_bytes = base64.b64decode(generated_response.data[0].b64_json)
generated_image = Image.open(io.BytesIO(generated_bytes)).convert("RGB")

generated_path = output_dir / "yandexart-exhibit.jpg"
generated_image.save(generated_path, format="JPEG", quality=95)

display(Markdown("### Художественная интерпретация YandexART"))
display(generated_image)
print(f"Файл сохранён: {generated_path}")

## Выводы

На одном музейном экспонате мы собрали полный мультимодальный конвейер Yandex AI Studio:

- Responses API поддержал простой запрос и связный диалог через `previous_response_id`;
- LLM превратила длинный научный текст в понятную детскую этикетку;
- Pydantic-схема сделала смысловые теги предсказуемыми и пригодными для программной обработки;
- VLM описала фотографию и вернула отдельный структурированный визуальный анализ;
- SpeechKit превратил доступное описание в музейный аудиофрагмент;
- YandexART создал новую художественную интерпретацию на основе полученного описания.

В реальном музейном продукте к этой схеме стоит добавить пакетную обработку всей коллекции,
кэширование результатов, журналирование версий моделей, автоматические проверки длины и
лексики, а главное — редакторскую верификацию фактов, атрибуции и доступного описания.
Текстовый анализ по музейной карточке обычно надёжнее визуальной гипотезы; расхождения между
`semantic_tags` и `visual_analysis` следует показывать куратору, а не скрывать.

Полезные ссылки:

- [Документация Yandex AI Studio](https://aistudio.yandex.ru/docs/)
- [Каталог моделей](https://aistudio.yandex.ru/docs/en/ai-studio/concepts/generation/models.html)
- страница первоисточника доступна в `exhibit["source_url"]` после загрузки датасета.